# UHI Modelling V2

I am attempting a revised workflow for modelling UHI.
First, the model is now going to include weather variables obtained from OpenMeteo. The data will then be aggregated with the Landsat data and used to build the model with the workflow below:  
1. Identify the world's climates from an authority (like the Trewartha climate classification).
2. Get substantial satellite (which will eventually include those those climate categories) and weather data on countries in those regions across the months (seasons) in a particular year.
3. Train a model on the data, implementing train-validation-test split, and predict UHI intensity on the test set.
4. Group the test examples by climate and compute the RMSE scores across the climates.
5. Cluster the UHI features to identify the types and categories of UHI.
6. Then I, human, assess the errors and prediction accuracies across each manufactured UHI cluster and how they vary across climates.

***"This project predicts urban heat island intensity using satellite and environmental data and evaluates how prediction reliability and errors vary across global climates and urban thermal types."***

## Testing aggregation with minimal data

To test the flow of data:  
1. From the various countries
2. In the various climates
3. Once a week from Jan 1 2025 to Dec 31 2025
4. From Earth Engine (Landsat) and OpenMeteo

I will be using as minimal data as possible. This will look like:  
1. Lagos, Ontario, Helsinki, and Tehran
2. Aw, Dc, Dcb, Bsk
3. Once a month Jan 1 2025 to Dec 31 2025
4. From Earth Engine and OpenMeteo

## Fetching from Open-Meteo

Again, the cities I want to sample are:
1. Lagos, Nigeria
2. Ontario, Canada
3. Tehran, Iran
4. Helsinki, Finland

The variables I am getting from Open-meteo are:
1. Cloud Cover (Low)
2. Air Temperature (2m)
3. Ralative Humidity
4. Precipitation
5. Wind Speed (10m)

In [2]:
import ee
import geopandas as gpd

ee.Authenticate()
ee.Initialize()

In [3]:
from spectral import get_spectral, get_viirs

cities = [
    {"code": "CAN", "name": "Ontario"},
    {"code": "IRN", "name": "Tehran"},
    {"code": "NGA", "name": "Lagos"},
	{"code": "FIN", "name": "Uusimaa"},
]

date_range = ("2025-01-01", "2025-12-31")

for city in cities:
    get_spectral(city['code'], city['name'], date_range)
    get_viirs(city['code'], date_range)


c:\Software Projects\Data Projects\uhi-modelling\uhenv\Lib\site-packages\geemap\conversion.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Sample points already exist at data/CAN_sample_points.csv.

VIIRS data already exists at data/CAN_viirs_features.csv.

Sample points already exist at data/IRN_sample_points.csv.

VIIRS data already exists at data/IRN_viirs_features.csv.

Sample points already exist at data/NGA_sample_points.csv.

VIIRS data already exists at data/NGA_viirs_features.csv.

Sample points already exist at data/FIN_sample_points.csv.

VIIRS data already exists at data/FIN_viirs_features.csv.



In [4]:
import pandas as pd

for city in cities:
    df = pd.read_csv(f"data/{city['code']}_sample_points.csv")
    urban = (df["LandCover"] == 50).sum()
    rural = (df["LandCover"] != 50).sum()
    print(f"{city['name']}: {urban} urban, {rural} rural, total: {len(df)}") 
    

Ontario: 100 urban, 100 rural, total: 200
Tehran: 100 urban, 100 rural, total: 200
Lagos: 100 urban, 100 rural, total: 200
Uusimaa: 100 urban, 100 rural, total: 200


In [5]:
for city in cities:
    df = pd.read_csv(f"data/{city['code']}_viirs_features.csv")
    print(f"{city['name']}: {len(df)} rows")
    print(df[["LST_day", "LST_night", "NDVI", "NDBI", "MNDWI", "SAVI", "Albedo"]].isnull().sum())

Ontario: 2400 rows
LST_day      0
LST_night    0
NDVI         0
NDBI         0
MNDWI        0
SAVI         0
Albedo       0
dtype: int64
Tehran: 2400 rows
LST_day      0
LST_night    0
NDVI         0
NDBI         0
MNDWI        0
SAVI         0
Albedo       0
dtype: int64
Lagos: 2229 rows
LST_day      0
LST_night    0
NDVI         0
NDBI         0
MNDWI        0
SAVI         0
Albedo       0
dtype: int64
Uusimaa: 2198 rows
LST_day      0
LST_night    0
NDVI         0
NDBI         0
MNDWI        0
SAVI         0
Albedo       0
dtype: int64


In [6]:
for city in cities:
    df = pd.read_csv(f"data/{city['code']}_viirs_features.csv")
    missing_months = set(range(1, 13)) - set(df["month"].unique())
    print(f"{city['name']}: missing months {missing_months}")

Ontario: missing months set()
Tehran: missing months set()
Lagos: missing months set()
Uusimaa: missing months {6}


In [7]:
from weather import process_city_weather

for city in cities:
    process_city_weather(city['code'], date_range)

Data has already been processed at data/CAN_Full_UHI_Data.csv.

Data has already been processed at data/IRN_Full_UHI_Data.csv.

Data has already been processed at data/NGA_Full_UHI_Data.csv.

Data has already been processed at data/FIN_Full_UHI_Data.csv.



In [8]:
for city in cities:
    df = pd.read_csv(f"data/{city['code']}_Full_UHI_Data.csv")
    print(f"{city['name']}: {len(df)} rows")

Ontario: 4800 rows
Tehran: 4800 rows
Lagos: 4800 rows
Uusimaa: 4800 rows


Now that we have fetched our data, we will add the `city` column and concatenate all four datasets.

In [9]:
# Step 1 — Consolidate all cities
viirs_dfs, weather_dfs = [], []

for city in cities:
    viirs_df = pd.read_csv(f"data/{city['code']}_viirs_features.csv")
    viirs_df["city"] = city["name"]
    viirs_dfs.append(viirs_df)

    weather_df = pd.read_csv(f"data/{city['code']}_Full_UHI_Data.csv")
    weather_df["city"] = city["name"]
    weather_dfs.append(weather_df)

viirs_data = pd.concat(viirs_dfs, ignore_index=True)
weather_data = pd.concat(weather_dfs, ignore_index=True)

# Step 2 — Parse date and extract month and time
weather_data["date"] = pd.to_datetime(weather_data["date"], utc=True)
weather_data["month"] = weather_data["date"].dt.month
weather_data["time"] = weather_data["date"].dt.hour.map({1: "night", 13: "day"})


# Step 4 — Merge VIIRS into uhi_data on lat/lon + city + month
uhi_data = weather_data.merge(
    viirs_data,
    on=["latitude", "longitude", "city", "month"],
    how="left"
)

print(uhi_data.shape)
print(uhi_data.head())
print(uhi_data[["LandCover", "Elevation", "LST_day", "LST_night", "NDVI", "NDBI", "MNDWI", "SAVI", "Albedo"]].isnull().sum())

(19200, 20)
                       date  humidity  precipitation  wind_speed  \
0 2025-01-01 01:00:00+00:00  96.11895            0.4   16.802786   
1 2025-01-01 13:00:00+00:00  88.68080            0.1   24.203810   
2 2025-02-01 01:00:00+00:00  86.77566            0.0   16.774803   
3 2025-02-01 13:00:00+00:00  80.08138            0.0   11.935778   
4 2025-03-01 01:00:00+00:00  74.14202            0.3   22.040997   

   cloud_cover_low  air_temperature   latitude  longitude  LandCover  \
0            100.0             1.20  43.431842 -80.482495       50.0   
1             65.0             0.35  43.431842 -80.482495       50.0   
2              0.0            -2.55  43.431842 -80.482495       50.0   
3              1.0           -15.65  43.431842 -80.482495       50.0   
4              1.0             2.80  43.431842 -80.482495       50.0   

    Elevation     city  month   time    Albedo   LST_day  LST_night     MNDWI  \
0  325.461426  Ontario      1  night  0.579363  5.272733   5.1836

In [10]:
print(uhi_data[uhi_data["LST_day"].isnull()]["city"].value_counts())
print(uhi_data[uhi_data["LST_day"].isnull()]["month"].value_counts())

city
Uusimaa    404
Lagos      342
Name: count, dtype: int64
month
6     426
3      48
8      36
7      30
12     28
4      28
9      28
2      26
10     26
5      26
11     26
1      18
Name: count, dtype: int64


In [11]:
uhi_data = uhi_data.sort_values(["city", "latitude", "longitude", "month", "time"]).reset_index(drop=True)

for col in ["LST_day", "LST_night", "NDVI", "NDBI", "MNDWI", "SAVI", "Albedo"]:
    uhi_data[col] = uhi_data.groupby(["city", "latitude", "longitude"])[col].transform(
        lambda x: x.interpolate(method="linear", limit_direction="both")
    )

print(uhi_data[["LST_day", "LST_night", "NDVI", "NDBI", "MNDWI", "SAVI", "Albedo"]].isnull().sum())

LST_day      192
LST_night    192
NDVI         192
NDBI         192
MNDWI        192
SAVI         192
Albedo       192
dtype: int64


In [12]:
null_mask = uhi_data["LST_day"].isnull()
print(uhi_data[null_mask]["city"].value_counts())
print(uhi_data[null_mask].groupby(["city", "latitude", "longitude"])["month"].apply(list))

city
Lagos    192
Name: count, dtype: int64
city   latitude  longitude
Lagos  6.478715  3.496877     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.479197  3.438773     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.497516  3.556839     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.503954  3.539145     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.512168  3.415794     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.515417  3.518704     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.517941  3.572257     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.577391  3.676986     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
Name: month, dtype: object


In [13]:
null_pixels = uhi_data[uhi_data["LST_day"].isnull()][["city", "latitude", "longitude"]].drop_duplicates()

uhi_data = uhi_data.merge(
    null_pixels.assign(drop=True),
    on=["city", "latitude", "longitude"],
    how="left"
)
uhi_data = uhi_data[uhi_data["drop"].isnull()].drop(columns=["drop"])

print(uhi_data.shape)
print(uhi_data[["LST_day", "LST_night"]].isnull().sum())
print(uhi_data["city"].value_counts())

(19008, 20)
LST_day      0
LST_night    0
dtype: int64
city
Ontario    4800
Tehran     4800
Uusimaa    4800
Lagos      4608
Name: count, dtype: int64


In [14]:
lagos = uhi_data[uhi_data["city"] == "Lagos"]
print(lagos[["latitude", "longitude", "LandCover"]].drop_duplicates()["LandCover"].value_counts())
print(f"Urban: {(lagos['LandCover'] == 50).sum() // 24}")
print(f"Rural: {(lagos['LandCover'] != 50).sum() // 24}")

LandCover
50.0    100
10.0     40
80.0     23
30.0     14
90.0      7
20.0      5
40.0      1
60.0      1
95.0      1
Name: count, dtype: int64
Urban: 100
Rural: 92


## Rest of the workflow.

In [15]:
uhi_data["is_urban"] = (uhi_data["LandCover"] == 50).astype(int)

urban = uhi_data[uhi_data["is_urban"] == 1].groupby(["city", "month", "time"]).agg(
    LST_day_urban=("LST_day", "mean"),
    LST_night_urban=("LST_night", "mean"),
    airtemp_urban=("air_temperature", "mean"),
    n_urban=("LST_day", "count")
).reset_index()

rural = uhi_data[uhi_data["is_urban"] == 0].groupby(["city", "month", "time"]).agg(
    LST_day_rural=("LST_day", "mean"),
    LST_night_rural=("LST_night", "mean"),
    airtemp_rural=("air_temperature", "mean"),
    n_rural=("LST_day", "count")
).reset_index()

uhi_intensity = urban.merge(rural, on=["city", "month", "time"], how="inner")

uhi_intensity["SUHI_day"] = uhi_intensity["LST_day_urban"] - uhi_intensity["LST_day_rural"]
uhi_intensity["SUHI_night"] = uhi_intensity["LST_night_urban"] - uhi_intensity["LST_night_rural"]
uhi_intensity["AUHI"] = uhi_intensity["airtemp_urban"] - uhi_intensity["airtemp_rural"]

uhi_intensity = uhi_intensity.drop(columns=[
    "LST_day_urban", "LST_day_rural",
    "LST_night_urban", "LST_night_rural",
    "airtemp_urban", "airtemp_rural"
])

print(uhi_intensity.shape)
print(uhi_intensity[["SUHI_day", "SUHI_night", "AUHI"]].describe())
print(uhi_intensity[["n_urban", "n_rural"]].value_counts())

(96, 8)
        SUHI_day  SUHI_night       AUHI
count  96.000000   96.000000  96.000000
mean    0.064075    0.040410   1.245613
std     0.072681    0.037440   1.958347
min    -0.058016   -0.008752  -3.187605
25%     0.014779    0.006326  -0.089142
50%     0.062932    0.027761   0.381726
75%     0.120517    0.081922   3.198875
max     0.210861    0.093826   7.611895
n_urban  n_rural
100      100        72
         92         24
Name: count, dtype: int64


# EDA

In [16]:
print(uhi_data.shape)
print(uhi_data.dtypes)
print(uhi_data.isnull().sum())
print(uhi_data.describe())

(19008, 21)
date               datetime64[ns, UTC]
humidity                       float64
precipitation                  float64
wind_speed                     float64
cloud_cover_low                float64
air_temperature                float64
latitude                       float64
longitude                      float64
LandCover                      float64
Elevation                      float64
city                            object
month                            int32
time                            object
Albedo                         float64
LST_day                        float64
LST_night                      float64
MNDWI                          float64
NDBI                           float64
NDVI                           float64
SAVI                           float64
is_urban                         int64
dtype: object
date               0
humidity           0
precipitation      0
wind_speed         0
cloud_cover_low    0
air_temperature    0
latitude           0
longitud

In [17]:
# 300K / 0.02 = 15000 — so raw values should be around 13000-16000
# Our values are 5-7, meaning raw values were ~250-350 (already in Kelvin in GEE)
print(uhi_data[["LST_day", "LST_night"]].describe())
print(uhi_data["LST_day"].mean() / 0.02)  # what the value should have been

            LST_day     LST_night
count  19008.000000  19008.000000
mean       5.932481      5.652698
std        0.328607      0.260875
min        5.147400      4.967015
25%        5.701567      5.429552
50%        5.981480      5.654038
75%        6.186600      5.909170
max        7.238000      6.208640
296.62403187658475


In [18]:
uhi_data["LST_day"] = uhi_data["LST_day"] / 0.02
uhi_data["LST_night"] = uhi_data["LST_night"] / 0.02

In [19]:
print(uhi_data[["LST_day", "LST_night"]].describe())

            LST_day     LST_night
count  19008.000000  19008.000000
mean     296.624032    282.634890
std       16.430366     13.043730
min      257.370000    248.350769
25%      285.078333    271.477624
50%      299.074000    282.701905
75%      309.330000    295.458500
max      361.900000    310.432000


In [20]:
urban = uhi_data[uhi_data["is_urban"] == 1].groupby(["city", "month", "time"]).agg(
    LST_day_urban=("LST_day", "mean"),
    LST_night_urban=("LST_night", "mean"),
    airtemp_urban=("air_temperature", "mean"),
    n_urban=("LST_day", "count")
).reset_index()

rural = uhi_data[uhi_data["is_urban"] == 0].groupby(["city", "month", "time"]).agg(
    LST_day_rural=("LST_day", "mean"),
    LST_night_rural=("LST_night", "mean"),
    airtemp_rural=("air_temperature", "mean"),
    n_rural=("LST_day", "count")
).reset_index()

uhi_intensity = urban.merge(rural, on=["city", "month", "time"], how="inner")
uhi_intensity["SUHI_day"] = uhi_intensity["LST_day_urban"] - uhi_intensity["LST_day_rural"]
uhi_intensity["SUHI_night"] = uhi_intensity["LST_night_urban"] - uhi_intensity["LST_night_rural"]
uhi_intensity["AUHI"] = uhi_intensity["airtemp_urban"] - uhi_intensity["airtemp_rural"]

uhi_intensity = uhi_intensity.drop(columns=[
    "LST_day_urban", "LST_day_rural",
    "LST_night_urban", "LST_night_rural",
    "airtemp_urban", "airtemp_rural"
])

print(uhi_intensity[["SUHI_day", "SUHI_night", "AUHI"]].describe())

        SUHI_day  SUHI_night       AUHI
count  96.000000   96.000000  96.000000
mean    3.203753    2.020487   1.245613
std     3.634073    1.872024   1.958347
min    -2.900799   -0.437594  -3.187605
25%     0.738942    0.316279  -0.089142
50%     3.146589    1.388053   0.381726
75%     6.025868    4.096086   3.198875
max    10.543050    4.691322   7.611895


In [22]:
print(uhi_intensity.sort_values(["city", "time", "month"]).to_string())

       city  month   time  n_urban  n_rural   SUHI_day  SUHI_night      AUHI
0     Lagos      1    day      100       92   6.206814    0.683552  1.126492
2     Lagos      2    day      100       92   7.460829    0.504271  0.394057
4     Lagos      3    day      100       92  10.543050    0.928984 -0.107986
6     Lagos      4    day      100       92   7.693824    0.882908 -0.103443
8     Lagos      5    day      100       92   7.288716   -0.019610  0.513188
10    Lagos      6    day      100       92   5.483313    0.650054  0.040275
12    Lagos      7    day      100       92   1.279126    0.651022  0.262275
14    Lagos      8    day      100       92   4.664551   -0.325409  0.466666
16    Lagos      9    day      100       92   6.256717    0.319340  0.256775
18    Lagos     10    day      100       92   6.422854    0.568262 -0.050660
20    Lagos     11    day      100       92   4.946484    0.316279  0.397623
22    Lagos     12    day      100       92   5.377758    0.025940  0.255862

In [23]:
predictors = uhi_data.groupby(["city", "month", "time"]).agg(
    NDVI=("NDVI", "mean"),
    NDBI=("NDBI", "mean"),
    MNDWI=("MNDWI", "mean"),
    SAVI=("SAVI", "mean"),
    Albedo=("Albedo", "mean"),
    LST_day=("LST_day", "mean"),
    LST_night=("LST_night", "mean"),
    humidity=("humidity", "mean"),
    precipitation=("precipitation", "mean"),
    wind_speed=("wind_speed", "mean"),
    cloud_cover_low=("cloud_cover_low", "mean"),
    air_temperature=("air_temperature", "mean"),
    Elevation=("Elevation", "mean")
).reset_index()

eda_df = uhi_intensity.merge(predictors, on=["city", "month", "time"], how="left")
print(eda_df.shape)
print(eda_df.columns.tolist())

(96, 21)
['city', 'month', 'time', 'n_urban', 'n_rural', 'SUHI_day', 'SUHI_night', 'AUHI', 'NDVI', 'NDBI', 'MNDWI', 'SAVI', 'Albedo', 'LST_day', 'LST_night', 'humidity', 'precipitation', 'wind_speed', 'cloud_cover_low', 'air_temperature', 'Elevation']


In [27]:
for city in ["Overall"] + ["Ontario", "Tehran", "Lagos", "Uusimaa"]:
    if city == "Overall":
        corr = eda_df[cols].corr()
    else:
        corr = eda_df[eda_df["city"] == city][cols].corr()
    
    print(f"\n{'='*20} {city} {'='*20}")
    print(corr[["SUHI_day", "SUHI_night", "AUHI"]].loc[["NDVI", "NDBI", "MNDWI", "SAVI", "Albedo", "humidity", "precipitation", "wind_speed", "cloud_cover_low", "air_temperature", "Elevation"]])


==================== Overall ====================
                 SUHI_day  SUHI_night      AUHI
NDVI             0.312651    0.052675 -0.122301
NDBI            -0.632824    0.454027  0.523830
MNDWI            0.386772   -0.488272 -0.439012
SAVI             0.378567    0.017492 -0.167892
Albedo           0.185634   -0.328134 -0.306233
humidity         0.531031   -0.651847 -0.649408
precipitation    0.188402   -0.185305 -0.081706
wind_speed       0.232775   -0.081086 -0.141085
cloud_cover_low  0.012032   -0.494755 -0.451074
air_temperature  0.095180   -0.106505 -0.008321
Elevation       -0.660069    0.800571  0.767811

==================== Ontario ====================
                 SUHI_day  SUHI_night      AUHI
NDVI             0.424797    0.722780 -0.071825
NDBI             0.320379   -0.211814 -0.261969
MNDWI           -0.475579   -0.525415  0.142791
SAVI             0.428085    0.716991 -0.075326
Albedo          -0.508969   -0.530842  0.078044
humidity        -0.161671    0.021

In [28]:
print("="*50)
print("MEAN UHI INTENSITY BY CITY AND TIME")
print("="*50)
print(uhi_intensity.groupby(["city", "time"])[["SUHI_day", "SUHI_night", "AUHI"]].mean().round(3))

print("\n" + "="*50)
print("DAY vs NIGHT DIFFERENCE PER CITY")
print("="*50)
for city in ["Ontario", "Tehran", "Lagos", "Uusimaa"]:
    day = uhi_intensity[(uhi_intensity["city"] == city) & (uhi_intensity["time"] == "day")]
    night = uhi_intensity[(uhi_intensity["city"] == city) & (uhi_intensity["time"] == "night")]
    print(f"\n{city}:")
    print(f"  SUHI_day   — Day: {day['SUHI_day'].mean():.3f}K   Night: {night['SUHI_day'].mean():.3f}K   Diff: {(day['SUHI_day'].mean() - night['SUHI_day'].mean()):.3f}K")
    print(f"  SUHI_night — Day: {day['SUHI_night'].mean():.3f}K  Night: {night['SUHI_night'].mean():.3f}K  Diff: {(day['SUHI_night'].mean() - night['SUHI_night'].mean()):.3f}K")
    print(f"  AUHI       — Day: {day['AUHI'].mean():.3f}°C  Night: {night['AUHI'].mean():.3f}°C  Diff: {(day['AUHI'].mean() - night['AUHI'].mean()):.3f}°C")

MEAN UHI INTENSITY BY CITY AND TIME
               SUHI_day  SUHI_night   AUHI
city    time                              
Lagos   day       6.135       0.432  0.288
        night     6.141       0.427 -0.083
Ontario day       6.141       3.128  0.225
        night     6.141       3.128  1.844
Tehran  day      -1.069       4.400  3.741
        night    -1.069       4.400  3.817
Uusimaa day       1.589       0.120  0.106
        night     1.621       0.128  0.028

DAY vs NIGHT DIFFERENCE PER CITY

Ontario:
  SUHI_day   — Day: 6.141K   Night: 6.141K   Diff: 0.000K
  SUHI_night — Day: 3.128K  Night: 3.128K  Diff: 0.000K
  AUHI       — Day: 0.225°C  Night: 1.844°C  Diff: -1.619°C

Tehran:
  SUHI_day   — Day: -1.069K   Night: -1.069K   Diff: 0.000K
  SUHI_night — Day: 4.400K  Night: 4.400K  Diff: 0.000K
  AUHI       — Day: 3.741°C  Night: 3.817°C  Diff: -0.075°C

Lagos:
  SUHI_day   — Day: 6.135K   Night: 6.141K   Diff: -0.005K
  SUHI_night — Day: 0.432K  Night: 0.427K  Diff: 0.005K
  AUHI  

In [29]:
uhi_intensity_wide = uhi_intensity.groupby(["city", "month"]).agg(
    n_urban=("n_urban", "first"),
    n_rural=("n_rural", "first"),
    SUHI_day=("SUHI_day", "mean"),
    SUHI_night=("SUHI_night", "mean"),
    AUHI_day=("AUHI", lambda x: x[uhi_intensity.loc[x.index, "time"] == "day"].values[0]),
    AUHI_night=("AUHI", lambda x: x[uhi_intensity.loc[x.index, "time"] == "night"].values[0])
).reset_index()

print(uhi_intensity_wide.shape)
print(uhi_intensity_wide.head())

(48, 8)
    city  month  n_urban  n_rural   SUHI_day  SUHI_night  AUHI_day  AUHI_night
0  Lagos      1      100       92   6.206814    0.683552  1.126492    0.121927
1  Lagos      2      100       92   7.471323    0.499926  0.394057    0.017623
2  Lagos      3      100       92  10.533344    0.921204 -0.107986   -0.274247
3  Lagos      4      100       92   7.687019    0.873726 -0.103443   -0.333530
4  Lagos      5      100       92   7.297396   -0.017030  0.513188   -0.127225


In [30]:
eda_df_wide = uhi_intensity_wide.merge(
    eda_df.groupby(["city", "month"]).agg(
        NDVI=("NDVI", "mean"),
        NDBI=("NDBI", "mean"),
        MNDWI=("MNDWI", "mean"),
        SAVI=("SAVI", "mean"),
        Albedo=("Albedo", "mean"),
        LST_day=("LST_day", "mean"),
        LST_night=("LST_night", "mean"),
        humidity=("humidity", "mean"),
        precipitation=("precipitation", "mean"),
        wind_speed=("wind_speed", "mean"),
        cloud_cover_low=("cloud_cover_low", "mean"),
        air_temperature=("air_temperature", "mean"),
        Elevation=("Elevation", "mean")
    ).reset_index(),
    on=["city", "month"],
    how="left"
)

print(eda_df_wide.shape)
print(eda_df_wide.columns.tolist())

(48, 21)
['city', 'month', 'n_urban', 'n_rural', 'SUHI_day', 'SUHI_night', 'AUHI_day', 'AUHI_night', 'NDVI', 'NDBI', 'MNDWI', 'SAVI', 'Albedo', 'LST_day', 'LST_night', 'humidity', 'precipitation', 'wind_speed', 'cloud_cover_low', 'air_temperature', 'Elevation']


In [31]:
cols_wide = ["SUHI_day", "SUHI_night", "AUHI_day", "AUHI_night",
             "NDVI", "NDBI", "MNDWI", "SAVI", "Albedo",
             "humidity", "precipitation", "wind_speed",
             "cloud_cover_low", "air_temperature", "Elevation"]

for city in ["Overall"] + ["Ontario", "Tehran", "Lagos", "Uusimaa"]:
    if city == "Overall":
        corr = eda_df_wide[cols_wide].corr()
    else:
        corr = eda_df_wide[eda_df_wide["city"] == city][cols_wide].corr()
    
    print(f"\n{'='*20} {city} {'='*20}")
    print(corr[["SUHI_day", "SUHI_night", "AUHI_day", "AUHI_night"]].loc[
        ["NDVI", "NDBI", "MNDWI", "SAVI", "Albedo", "humidity", 
         "precipitation", "wind_speed", "cloud_cover_low", "air_temperature", "Elevation"]
    ].round(3))


==================== Overall ====================
                 SUHI_day  SUHI_night  AUHI_day  AUHI_night
NDVI                0.313       0.053    -0.106      -0.137
NDBI               -0.633       0.454     0.667       0.416
MNDWI               0.387      -0.488    -0.559      -0.349
SAVI                0.379       0.017    -0.156      -0.179
Albedo              0.186      -0.328    -0.434      -0.207
humidity            0.581      -0.714    -0.791      -0.656
precipitation       0.248      -0.244    -0.226      -0.137
wind_speed          0.262      -0.091    -0.315      -0.055
cloud_cover_low     0.013      -0.534    -0.523      -0.436
air_temperature     0.100      -0.111     0.111      -0.164
Elevation          -0.660       0.801     0.880       0.689

==================== Ontario ====================
                 SUHI_day  SUHI_night  AUHI_day  AUHI_night
NDVI                0.425       0.723     0.186      -0.224
NDBI                0.320      -0.212    -0.037      -0.43

In [32]:
uhi_data_model = uhi_data.merge(
    uhi_intensity_wide[["city", "month", "SUHI_day", "SUHI_night", "AUHI_day", "AUHI_night"]],
    on=["city", "month"],
    how="left"
)

print(uhi_data_model.shape)
print(uhi_data_model[["SUHI_day", "SUHI_night", "AUHI_day", "AUHI_night"]].isnull().sum())

(19008, 25)
SUHI_day      0
SUHI_night    0
AUHI_day      0
AUHI_night    0
dtype: int64


In [33]:
print(uhi_data_model.head())
print(uhi_data_model.info())

                       date   humidity  precipitation  wind_speed  \
0 2025-01-01 13:00:00+00:00  65.052124            0.1   11.126562   
1 2025-01-01 01:00:00+00:00  93.391365            0.0    7.895416   
2 2025-02-01 13:00:00+00:00  64.541010            0.0   12.251905   
3 2025-02-01 01:00:00+00:00  89.099350            0.0   11.085720   
4 2025-03-01 13:00:00+00:00  67.474920            0.1   13.841286   

   cloud_cover_low  air_temperature  latitude  longitude  LandCover  \
0             15.0        31.256500   6.39659   2.712946       30.0   
1              6.0        25.956501   6.39659   2.712946       30.0   
2             14.0        31.606500   6.39659   2.712946       30.0   
3              7.0        26.956501   6.39659   2.712946       30.0   
4             13.0        31.456501   6.39659   2.712946       30.0   

   Elevation  ...   LST_night     MNDWI      NDBI      NDVI      SAVI  \
0   2.860185  ...  296.867500 -0.145213 -0.188529  0.326602  0.232950   
1   2.860185

## Initial Modelling

In [34]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Step 1 — Select relevant columns
features = ["humidity", "precipitation", "wind_speed", "cloud_cover_low",
            "air_temperature", "latitude", "longitude", "Elevation",
            "Albedo", "MNDWI", "NDBI", "SAVI", "LandCover", "month"]

targets = ["SUHI_day", "SUHI_night", "AUHI_day", "AUHI_night"]

# Step 2 — Build modelling dataframe
model_df = uhi_data_model[features + targets].copy()

# Step 3 — Encode LandCover as category then one-hot encode
model_df["LandCover"] = model_df["LandCover"].astype(int).astype(str)
model_df = pd.get_dummies(model_df, columns=["LandCover"], prefix="LC")

# Step 4 — Encode month as category (cyclical encoding)
# Month is cyclical — December and January are adjacent
model_df["month_sin"] = np.sin(2 * np.pi * model_df["month"] / 12)
model_df["month_cos"] = np.cos(2 * np.pi * model_df["month"] / 12)
model_df = model_df.drop(columns=["month"])

print(model_df.shape)
print(model_df.dtypes)
print(model_df.head())

(19008, 28)
humidity           float64
precipitation      float64
wind_speed         float64
cloud_cover_low    float64
air_temperature    float64
latitude           float64
longitude          float64
Elevation          float64
Albedo             float64
MNDWI              float64
NDBI               float64
SAVI               float64
SUHI_day           float64
SUHI_night         float64
AUHI_day           float64
AUHI_night         float64
LC_10                 bool
LC_100                bool
LC_20                 bool
LC_30                 bool
LC_40                 bool
LC_50                 bool
LC_60                 bool
LC_80                 bool
LC_90                 bool
LC_95                 bool
month_sin          float64
month_cos          float64
dtype: object
    humidity  precipitation  wind_speed  cloud_cover_low  air_temperature  \
0  65.052124            0.1   11.126562             15.0        31.256500   
1  93.391365            0.0    7.895416              6.0        

In [35]:
model_df.to_csv("data/model_df.csv", index = False)

In [36]:
from sklearn.model_selection import train_test_split

feature_cols = [c for c in model_df.columns if c not in targets]

X = model_df[feature_cols]
y = model_df[targets]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

X_train: (15206, 24)
X_test:  (3802, 24)
y_train: (15206, 4)
y_test:  (3802, 4)


In [37]:
from sklearn.model_selection import train_test_split

# Get unique pixels
unique_pixels = uhi_data_model[["latitude", "longitude"]].drop_duplicates().reset_index(drop=True)

# Split pixels into train and test
train_pixels, test_pixels = train_test_split(unique_pixels, test_size=0.2, random_state=21)

# Create pixel identifier in model_df
model_df["latitude"] = uhi_data_model["latitude"]
model_df["longitude"] = uhi_data_model["longitude"]

# Split based on pixel membership
train_mask = model_df[["latitude", "longitude"]].apply(tuple, axis=1).isin(
    train_pixels.apply(tuple, axis=1)
)

X_train = model_df[train_mask][feature_cols]
X_test = model_df[~train_mask][feature_cols]
y_train = model_df[train_mask][targets]
y_test = model_df[~train_mask][targets]

# Drop lat/lon from features if they leaked in
X_train = X_train[feature_cols]
X_test = X_test[feature_cols]

print(f"Train pixels: {len(train_pixels)}, Test pixels: {len(test_pixels)}")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

Train pixels: 633, Test pixels: 159
X_train: (15192, 24)
X_test:  (3816, 24)
y_train: (15192, 4)
y_test:  (3816, 4)


### Creating the first models

In [38]:
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import numpy as np

models = {
    "Linear Regression": MultiOutputRegressor(LinearRegression()),
    "Ridge Regression": MultiOutputRegressor(Ridge(alpha=1.0)),
    "Random Forest": MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=21)),
    "Gradient Boosting": MultiOutputRegressor(GradientBoostingRegressor(n_estimators=100, random_state=21)),
    "XGBoost": MultiOutputRegressor(XGBRegressor(n_estimators=100, random_state=21, verbosity=0))
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    model_results = {}
    for i, target in enumerate(targets):
        r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_test.iloc[:, i], y_pred[:, i]))
        model_results[target] = {"R2": round(r2, 3), "MAE": round(mae, 3), "RMSE": round(rmse, 3)}
    
    results[name] = model_results
    print("Done.")

# Print results table
for name, model_results in results.items():
    print(f"\n{'='*20} {name} {'='*20}")
    print(f"{'Target':<15} {'R2':>8} {'MAE':>8} {'RMSE':>8}")
    for target, metrics in model_results.items():
        print(f"{target:<15} {metrics['R2']:>8} {metrics['MAE']:>8} {metrics['RMSE']:>8}")

Training Linear Regression...
Done.
Training Ridge Regression...
Done.
Training Random Forest...
Done.
Training Gradient Boosting...
Done.
Training XGBoost...
Done.

==================== Linear Regression ====================
Target                R2      MAE     RMSE
SUHI_day           0.842    1.146     1.47
SUHI_night         0.842    0.591    0.762
AUHI_day           0.779    0.631    0.858
AUHI_night         0.618    0.886    1.332

==================== Ridge Regression ====================
Target                R2      MAE     RMSE
SUHI_day           0.842    1.154    1.473
SUHI_night         0.841    0.595    0.762
AUHI_day           0.779    0.628    0.858
AUHI_night         0.618    0.872    1.332

==================== Random Forest ====================
Target                R2      MAE     RMSE
SUHI_day           0.999    0.012    0.084
SUHI_night           1.0    0.002    0.014
AUHI_day             1.0    0.001    0.012
AUHI_night           1.0    0.004     0.04

===========

In [39]:
results_overfit = {}

for name, model in models.items():
    print(f"Evaluating {name}...")
    
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    model_results = {}
    for i, target in enumerate(targets):
        r2_train = r2_score(y_train.iloc[:, i], y_pred_train[:, i])
        r2_test = r2_score(y_test.iloc[:, i], y_pred_test[:, i])
        mae_train = mean_absolute_error(y_train.iloc[:, i], y_pred_train[:, i])
        mae_test = mean_absolute_error(y_test.iloc[:, i], y_pred_test[:, i])
        
        model_results[target] = {
            "R2_train": round(r2_train, 3),
            "R2_test": round(r2_test, 3),
            "R2_gap": round(r2_train - r2_test, 3),
            "MAE_train": round(mae_train, 3),
            "MAE_test": round(mae_test, 3),
            "MAE_gap": round(mae_test - mae_train, 3)
        }
    
    results_overfit[name] = model_results

for name, model_results in results_overfit.items():
    print(f"\n{'='*20} {name} {'='*20}")
    print(f"{'Target':<15} {'R2_train':>10} {'R2_test':>10} {'R2_gap':>10} {'MAE_train':>10} {'MAE_test':>10} {'MAE_gap':>10}")
    for target, metrics in model_results.items():
        print(f"{target:<15} {metrics['R2_train']:>10} {metrics['R2_test']:>10} {metrics['R2_gap']:>10} {metrics['MAE_train']:>10} {metrics['MAE_test']:>10} {metrics['MAE_gap']:>10}")

Evaluating Linear Regression...
Evaluating Ridge Regression...
Evaluating Random Forest...
Evaluating Gradient Boosting...
Evaluating XGBoost...

==================== Linear Regression ====================
Target            R2_train    R2_test     R2_gap  MAE_train   MAE_test    MAE_gap
SUHI_day             0.829      0.842     -0.013      1.147      1.146     -0.001
SUHI_night           0.838      0.842     -0.003      0.564      0.591      0.026
AUHI_day             0.729      0.779      -0.05      0.645      0.631     -0.014
AUHI_night           0.582      0.618     -0.036      0.905      0.886     -0.019

==================== Ridge Regression ====================
Target            R2_train    R2_test     R2_gap  MAE_train   MAE_test    MAE_gap
SUHI_day             0.827      0.842     -0.014      1.158      1.154     -0.003
SUHI_night           0.836      0.841     -0.005      0.571      0.595      0.024
AUHI_day             0.729      0.779      -0.05      0.644      0.628     -0.

## Attempting to recompute the SUHI and AUHI

The previous approach calculated the difference between the mean of urban LSTs and the mean of rural LSTs. This would've caused some data leakage, as all points within the same state would've had the same value, and could be the cause of the suspiciously good performances of the model.
The new approach would calculate the difference between each point's LST and the mean rural LST.

In [40]:
# Compute mean rural LST per city per month
rural_means = uhi_data[uhi_data["is_urban"] == 0].groupby(["city", "month"]).agg(
    rural_mean_LST_day=("LST_day", "mean"),
    rural_mean_LST_night=("LST_night", "mean"),
    rural_mean_airtemp=("air_temperature", "mean")
).reset_index()

# Merge back onto full dataset
uhi_data = uhi_data.merge(rural_means, on=["city", "month"], how="left")

# Compute pixel-level SUHI and AUHI
uhi_data["SUHI_day"] = uhi_data["LST_day"] - uhi_data["rural_mean_LST_day"]
uhi_data["SUHI_night"] = uhi_data["LST_night"] - uhi_data["rural_mean_LST_night"]
uhi_data["AUHI"] = uhi_data["air_temperature"] - uhi_data["rural_mean_airtemp"]

print(uhi_data[["SUHI_day", "SUHI_night", "AUHI"]].describe())
print(uhi_data[["SUHI_day", "SUHI_night", "AUHI"]].isnull().sum())

           SUHI_day    SUHI_night          AUHI
count  19008.000000  19008.000000  19008.000000
mean       1.618057      1.020448      0.629097
std        5.142381      2.943498      4.536121
min      -37.308680    -18.307601    -22.360505
25%       -1.194788     -0.659445     -1.843040
50%        1.257782      0.829390      0.160970
75%        4.783755      2.896015      2.353266
max       52.101830     13.251053     14.637495
SUHI_day      0
SUHI_night    0
AUHI          0
dtype: int64


Rebuilding the models with new data:

In [41]:
targets = ["SUHI_day", "SUHI_night", "AUHI"]

model_df = uhi_data[features + targets].copy()

# Re-encode
model_df["LandCover"] = model_df["LandCover"].astype(int).astype(str)
model_df = pd.get_dummies(model_df, columns=["LandCover"], prefix="LC")
model_df["month_sin"] = np.sin(2 * np.pi * model_df["month"] / 12)
model_df["month_cos"] = np.cos(2 * np.pi * model_df["month"] / 12)
model_df = model_df.drop(columns=["month"])

# Add lat/lon back from uhi_data for pixel-level split
model_df["latitude"] = uhi_data["latitude"]
model_df["longitude"] = uhi_data["longitude"]

# Pixel-level train/test split
unique_pixels = uhi_data[["latitude", "longitude"]].drop_duplicates().reset_index(drop=True)
train_pixels, test_pixels = train_test_split(unique_pixels, test_size=0.2, random_state=42)

train_mask = model_df[["latitude", "longitude"]].apply(tuple, axis=1).isin(
    train_pixels.apply(tuple, axis=1)
)

feature_cols = [c for c in model_df.columns if c not in targets + ["latitude", "longitude"]]

X_train = model_df[train_mask][feature_cols]
X_test = model_df[~train_mask][feature_cols]
y_train = model_df[train_mask][targets]
y_test = model_df[~train_mask][targets]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

X_train: (15192, 22), X_test: (3816, 22)
y_train: (15192, 3), y_test: (3816, 3)


In [42]:
models = {
    "Linear Regression": MultiOutputRegressor(LinearRegression()),
    "Ridge Regression": MultiOutputRegressor(Ridge(alpha=1.0)),
    "Random Forest": MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": MultiOutputRegressor(GradientBoostingRegressor(n_estimators=100, random_state=42)),
    "XGBoost": MultiOutputRegressor(XGBRegressor(n_estimators=100, random_state=42, verbosity=0))
}

results_overfit = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    model_results = {}
    for i, target in enumerate(targets):
        r2_train = r2_score(y_train.iloc[:, i], y_pred_train[:, i])
        r2_test = r2_score(y_test.iloc[:, i], y_pred_test[:, i])
        mae_train = mean_absolute_error(y_train.iloc[:, i], y_pred_train[:, i])
        mae_test = mean_absolute_error(y_test.iloc[:, i], y_pred_test[:, i])
        
        model_results[target] = {
            "R2_train": round(r2_train, 3),
            "R2_test": round(r2_test, 3),
            "R2_gap": round(r2_train - r2_test, 3),
            "MAE_train": round(mae_train, 3),
            "MAE_test": round(mae_test, 3),
            "MAE_gap": round(mae_test - mae_train, 3)
        }
    
    results_overfit[name] = model_results
    print(f"Done.")

for name, model_results in results_overfit.items():
    print(f"\n{'='*20} {name} {'='*20}")
    print(f"{'Target':<15} {'R2_train':>10} {'R2_test':>10} {'R2_gap':>10} {'MAE_train':>10} {'MAE_test':>10} {'MAE_gap':>10}")
    for target, metrics in model_results.items():
        print(f"{target:<15} {metrics['R2_train']:>10} {metrics['R2_test']:>10} {metrics['R2_gap']:>10} {metrics['MAE_train']:>10} {metrics['MAE_test']:>10} {metrics['MAE_gap']:>10}")

Training Linear Regression...
Done.
Training Ridge Regression...
Done.
Training Random Forest...
Done.
Training Gradient Boosting...
Done.
Training XGBoost...
Done.

==================== Linear Regression ====================
Target            R2_train    R2_test     R2_gap  MAE_train   MAE_test    MAE_gap
SUHI_day             0.481      0.491     -0.011      2.779      2.865      0.086
SUHI_night           0.385      0.402     -0.017      1.748      1.893      0.146
AUHI                 0.489      0.513     -0.025      2.484      2.487      0.003

==================== Ridge Regression ====================
Target            R2_train    R2_test     R2_gap  MAE_train   MAE_test    MAE_gap
SUHI_day             0.481      0.491      -0.01      2.779      2.866      0.086
SUHI_night           0.383      0.401     -0.019      1.746      1.891      0.145
AUHI                 0.489      0.513     -0.025      2.484      2.487      0.003

==================== Random Forest ====================
T

In [44]:
print(model_df.head())
model_df.to_csv("data/model2_df.csv", index = False)

    humidity  precipitation  wind_speed  cloud_cover_low  air_temperature  \
0  65.052124            0.1   11.126562             15.0        31.256500   
1  93.391365            0.0    7.895416              6.0        25.956501   
2  64.541010            0.0   12.251905             14.0        31.606500   
3  89.099350            0.0   11.085720              7.0        26.956501   
4  67.474920            0.1   13.841286             13.0        31.456501   

   latitude  longitude  Elevation    Albedo     MNDWI  ...  LC_20  LC_30  \
0   6.39659   2.712946   2.860185  0.198264 -0.145213  ...  False   True   
1   6.39659   2.712946   2.860185  0.198264 -0.145213  ...  False   True   
2   6.39659   2.712946   2.860185  0.219711 -0.126637  ...  False   True   
3   6.39659   2.712946   2.860185  0.219711 -0.126637  ...  False   True   
4   6.39659   2.712946   2.860185  0.207461 -0.173548  ...  False   True   

   LC_40  LC_50  LC_60  LC_80  LC_90  LC_95  month_sin     month_cos  
0  False 

In [45]:
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import numpy as np

models = {
    "Linear Regression": MultiOutputRegressor(LinearRegression()),
    "Ridge Regression": MultiOutputRegressor(Ridge(alpha=1.0)),
    "Random Forest": MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": MultiOutputRegressor(GradientBoostingRegressor(n_estimators=100, random_state=42)),
    "XGBoost": MultiOutputRegressor(XGBRegressor(n_estimators=100, random_state=42, verbosity=0))
}

holdout_results = {}

for holdout_city in ["Ontario", "Tehran", "Lagos", "Uusimaa"]:
    print(f"\nHoldout city: {holdout_city}")
    
    train_mask = uhi_data["city"] != holdout_city
    test_mask = uhi_data["city"] == holdout_city
    
    X_train_ho = model_df[train_mask][feature_cols]
    X_test_ho = model_df[test_mask][feature_cols]
    y_train_ho = model_df[train_mask][targets]
    y_test_ho = model_df[test_mask][targets]
    
    city_results = {}
    
    for name, model in models.items():
        model.fit(X_train_ho, y_train_ho)
        y_pred = model.predict(X_test_ho)
        
        model_results = {}
        for i, target in enumerate(targets):
            r2 = r2_score(y_test_ho.iloc[:, i], y_pred[:, i])
            mae = mean_absolute_error(y_test_ho.iloc[:, i], y_pred[:, i])
            model_results[target] = {"R2": round(r2, 3), "MAE": round(mae, 3)}
        
        city_results[name] = model_results
    
    holdout_results[holdout_city] = city_results

# Print results
for holdout_city, city_results in holdout_results.items():
    print(f"\n{'='*20} Holdout: {holdout_city} {'='*20}")
    for name, model_results in city_results.items():
        print(f"\n  {name}:")
        print(f"  {'Target':<15} {'R2':>8} {'MAE':>8}")
        for target, metrics in model_results.items():
            print(f"  {target:<15} {metrics['R2']:>8} {metrics['MAE']:>8}")


Holdout city: Ontario

Holdout city: Tehran

Holdout city: Lagos

Holdout city: Uusimaa

==================== Holdout: Ontario ====================

  Linear Regression:
  Target                R2      MAE
  SUHI_day           0.186      3.3
  SUHI_night        -0.157    2.288
  AUHI               -0.64      3.4

  Ridge Regression:
  Target                R2      MAE
  SUHI_day           0.185    3.299
  SUHI_night        -0.161    2.287
  AUHI              -0.633    3.396

  Random Forest:
  Target                R2      MAE
  SUHI_day          -0.367    4.418
  SUHI_night        -0.842    2.911
  AUHI              -0.099    2.716

  Gradient Boosting:
  Target                R2      MAE
  SUHI_day          -0.126    3.956
  SUHI_night        -0.612    2.696
  AUHI              -0.012    2.493

  XGBoost:
  Target                R2      MAE
  SUHI_day          -0.227    4.168
  SUHI_night         -0.86    2.945
  AUHI              -0.028     2.54

==================== Holdout: Tehra

## Tuning hyperparameters using Grid Search Cross Validation

In [46]:
from sklearn.model_selection import GridSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np
import json

rf_params = {
    "estimator__n_estimators": [100, 200, 300],
    "estimator__max_depth": [5, 10, 15, None],
    "estimator__min_samples_split": [2, 5, 10],
    "estimator__min_samples_leaf": [1, 2, 4],
    "estimator__max_features": ["sqrt", "log2"]
}

gb_params = {
    "estimator__n_estimators": [100, 200, 300],
    "estimator__max_depth": [3, 5, 7],
    "estimator__learning_rate": [0.01, 0.05, 0.1],
    "estimator__min_samples_split": [2, 5, 10],
    "estimator__subsample": [0.8, 0.9, 1.0]
}

xgb_params = {
    "estimator__n_estimators": [100, 200, 300],
    "estimator__max_depth": [3, 5, 7],
    "estimator__learning_rate": [0.01, 0.05, 0.1],
    "estimator__subsample": [0.8, 0.9, 1.0],
    "estimator__colsample_bytree": [0.8, 0.9, 1.0],
    "estimator__reg_alpha": [0, 0.1, 0.5],
    "estimator__reg_lambda": [1, 1.5, 2]
}

tuned_models = {
    "Linear Regression": MultiOutputRegressor(LinearRegression()),
    "Ridge Regression": MultiOutputRegressor(Ridge(alpha=1.0))
}

for name, base_model, params in [
    ("Random Forest", MultiOutputRegressor(RandomForestRegressor(random_state=42)), rf_params),
    ("Gradient Boosting", MultiOutputRegressor(GradientBoostingRegressor(random_state=42)), gb_params),
    ("XGBoost", MultiOutputRegressor(XGBRegressor(random_state=42, verbosity=0)), xgb_params)
]:
    print(f"Tuning {name}...")
    grid_search = GridSearchCV(
        base_model,
        params,
        cv=5,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(X_train, y_train)
    tuned_models[name] = grid_search.best_estimator_
    print(f"[{name}] Best params: {grid_search.best_params_}")
    print(f"[{name}] Best CV R2: {grid_search.best_score_:.3f}\n")

# Fit linear models directly — no tuning needed
tuned_models["Linear Regression"].fit(X_train, y_train)
tuned_models["Ridge Regression"].fit(X_train, y_train)

# Evaluate all tuned models
print("\n" + "="*60)
print("TUNED MODEL PERFORMANCE")
print("="*60)

for name, model in tuned_models.items():
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    print(f"\n{'='*20} {name} {'='*20}")
    print(f"{'Target':<15} {'R2_train':>10} {'R2_test':>10} {'R2_gap':>10} {'MAE_train':>10} {'MAE_test':>10}")
    for i, target in enumerate(targets):
        r2_train = r2_score(y_train.iloc[:, i], y_pred_train[:, i])
        r2_test = r2_score(y_test.iloc[:, i], y_pred_test[:, i])
        mae_train = mean_absolute_error(y_train.iloc[:, i], y_pred_train[:, i])
        mae_test = mean_absolute_error(y_test.iloc[:, i], y_pred_test[:, i])
        print(f"{target:<15} {r2_train:>10.3f} {r2_test:>10.3f} {r2_train-r2_test:>10.3f} {mae_train:>10.3f} {mae_test:>10.3f}")


# Collect results into a dictionary
tuned_results = {}

for name, model in tuned_models.items():
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    model_results = {}
    for i, target in enumerate(targets):
        r2_train = r2_score(y_train.iloc[:, i], y_pred_train[:, i])
        r2_test = r2_score(y_test.iloc[:, i], y_pred_test[:, i])
        mae_train = mean_absolute_error(y_train.iloc[:, i], y_pred_train[:, i])
        mae_test = mean_absolute_error(y_test.iloc[:, i], y_pred_test[:, i])
        rmse_test = np.sqrt(mean_squared_error(y_test.iloc[:, i], y_pred_test[:, i]))
        
        model_results[target] = {
            "R2_train": round(r2_train, 3),
            "R2_test": round(r2_test, 3),
            "R2_gap": round(r2_train - r2_test, 3),
            "MAE_train": round(mae_train, 3),
            "MAE_test": round(mae_test, 3),
            "RMSE_test": round(rmse_test, 3)
        }
    
    tuned_results[name] = model_results

# Save to JSON
with open("results/tuned_model_results.json", "w") as f:
    json.dump(tuned_results, f, indent=4)

# Save to CSV for easier reading
rows = []
for name, model_results in tuned_results.items():
    for target, metrics in model_results.items():
        rows.append({"model": name, "target": target, **metrics})

pd.DataFrame(rows).to_csv("results/tuned_model_results.csv", index=False)
print("Results saved to results/tuned_model_results.json and results/tuned_model_results.csv")


best_params = {}
for name, model in tuned_models.items():
    if hasattr(model, "estimators_"):
        # Extract params from first estimator as representative
        best_params[name] = model.estimators_[0].get_params()

with open("results/best_params.json", "w") as f:
    json.dump(best_params, f, indent=4)
print("Best params saved to results/best_params.json")

Tuning Random Forest...
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
[Random Forest] Best params: {'estimator__max_depth': None, 'estimator__max_features': 'sqrt', 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 2, 'estimator__n_estimators': 300}
[Random Forest] Best CV R2: 0.112

Tuning Gradient Boosting...
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
[Gradient Boosting] Best params: {'estimator__learning_rate': 0.05, 'estimator__max_depth': 5, 'estimator__min_samples_split': 10, 'estimator__n_estimators': 300, 'estimator__subsample': 0.8}
[Gradient Boosting] Best CV R2: 0.069

Tuning XGBoost...
Fitting 5 folds for each of 2187 candidates, totalling 10935 fits
[XGBoost] Best params: {'estimator__colsample_bytree': 0.9, 'estimator__learning_rate': 0.05, 'estimator__max_depth': 7, 'estimator__n_estimators': 300, 'estimator__reg_alpha': 0.1, 'estimator__reg_lambda': 1, 'estimator__subsample': 0.8}
[XGBoost] Best CV R2: 0.105


TUNED